# Fig S5A — GWAS design with population mismatch (data processing)

Target population is GTEx unrelated EUR (subset); references are ADSP / 1000 Genomes. Power / FDR of SuSiE-RSS and SuSiE-rRSS (EB-mix with SER fallback). Bootstrap 95% CIs over replicates.

**Output:** `figure_data/FigS5A.GWAS_Design_Benchmark.rds` (+ `.csv`)

In [1]:
source("../../R/utils.R")

data_name <- "FigS5A.GWAS_Design_Benchmark"

susie <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/GWAS_fine/summary/result_susie_bias_summary_summary.rds")

In [2]:
## ---- SuSiE-rRSS = EB-mix, falling back to SER when the reliability flag is TRUE
count_cols <- c("true_var_number", "true_set_number",
                "total_var_number", "total_set_number", "total_set_length")
ser_suffix <- "_mix_SER"        # <- SER partner for the fallback; use "_SER" for plain SER

## EB-mix rows (carry the reliability flag) and their SER partners, keyed by LD ref
mix_rows <- susie %>%
  filter(str_detect(method, "_mix$")) %>%
  mutate(ldref = str_remove(method, "_mix$"))

ser_rows <- susie %>%
  filter(str_detect(method, paste0(ser_suffix, "$"))) %>%
  mutate(ldref = str_remove(method, paste0(ser_suffix, "$")))

fallback <- mix_rows %>%
  select(input_file, ncausal, ldref, R_reliability_flag, all_of(count_cols)) %>%
  rename_with(~ paste0(.x, "_mix"), all_of(count_cols)) %>%
  left_join(
    ser_rows %>% select(input_file, ldref, all_of(count_cols)) %>%
      rename_with(~ paste0(.x, "_ser"), all_of(count_cols)),
    by = c("input_file", "ldref")
  ) %>%
  mutate(use_ser = R_reliability_flag %in% TRUE) %>%       # TRUE (& non-NA) -> unreliable -> SER
  transmute(
    input_file,
    method  = paste0(ldref, "_eb_mix_SER_fallback"),        # matches the GTEx naming
    ncausal,
    true_var_number  = if_else(use_ser & !is.na(true_var_number_ser),  true_var_number_ser,  true_var_number_mix),
    true_set_number  = if_else(use_ser & !is.na(true_set_number_ser),  true_set_number_ser,  true_set_number_mix),
    total_var_number = if_else(use_ser & !is.na(total_var_number_ser), total_var_number_ser, total_var_number_mix),
    total_set_number = if_else(use_ser & !is.na(total_set_number_ser), total_set_number_ser, total_set_number_mix),
    total_set_length = if_else(use_ser & !is.na(total_set_length_ser), total_set_length_ser, total_set_length_mix)
  )

## sanity checks
stopifnot(nrow(fallback) == nrow(mix_rows))          # one fallback row per mix row
stopifnot(!any(is.na(fallback$total_var_number)))    # every fallback resolved to a real row

susie <- bind_rows(susie, fallback)                  # append; missing diagnostic cols -> NA

In [3]:
susie_std <- susie %>%
  mutate(panel_raw = str_extract(method, "^[A-Za-z]+"),
         variant   = str_remove(method, "^[A-Za-z]+")) %>%
  mutate(panel = recode(panel_raw, insample = "gtex"),
         qc = case_when(
           variant == ""                     ~ "susie",
           variant == "_mix"                 ~ "susie_eb_mix",
           variant == "_eb_mix_SER_fallback" ~ "susie_eb_SER_fallback",
           TRUE                              ~ NA_character_)) %>%
  filter(!is.na(qc)) %>%
  mutate(method  = panel,
         ncausal = as.integer(str_extract(input_file, "(?<=causal_)\\d+"))) %>%
  select(input_file, method, ncausal, true_var_number, true_set_number,
         total_var_number, total_set_number, total_set_length, qc)

cell_reps <- susie_std %>%
  mutate(h2g = str_extract(input_file, "h2g_[0-9.]+"),
         causal_k = total_var_number)          # alias survives group_modify

## ---- bootstrap 95% CIs (boot_cell() in R/utils.R) ---------------------------
n_boot <- 1000
set.seed(1)

summary_ci <- cell_reps %>%
  group_by(qc, method, h2g, total_var_number) %>%
  group_modify(~ boot_cell(.x, B = n_boot)) %>%
  ungroup()

In [5]:

n_gwas <- 50000                                 # <-- VERIFY; NCP = n * h2g

plot_df <- summary_ci %>%
  mutate(
    method = recode(method,
      gtex = "In-Sample", ADSP = "ADSP", UKB = "UKB", kg = "1000 Genomes",
      .default = as.character(method)),
    method = factor(method, levels = c("In-Sample", "ADSP", "UKB", "1000 Genomes")),
    h2g_num = as.numeric(str_remove(h2g, "h2g_")),
    ncp_num = n_gwas * h2g_num,
    ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num),
    qc = recode(qc,
      susie                 = "SuSiE-RSS",
      susie_eb_mix          = "SuSiE-rRSS -SER",
      susie_eb_SER_fallback = "SuSiE-rRSS",
      .default = as.character(qc)),
    qc = factor(qc, levels = names(pal_qc))
  ) %>%
  filter(qc != "DENTIST", ncp_num != 25)        # also drops "SuSiE-rRSS -SER" (NA), as in Fig 3A
stopifnot(!any(is.na(plot_df$qc)))

save_data(plot_df, data_name)